# ML-09 — Validation and Research Claim Audit

This notebook audits the methodology of the FlyRank Research Paper (`docs/flyrank-seo-research-march-2026.pdf`) and applies the same validation rigor to our **Lane 2: Refresh / Content Opportunity Scoring** model.

> Skill loaded: `skills/hunting-leakage-and-validating/SKILL.md`, `skills/flyrank/flyrank-data/SKILL.md`, & `skills/writing-honest-claims/SKILL.md`

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from? Does the validation design support the claim? Frame questions respectfully and concretely.*

### Methodology Questions on FlyRank Research Paper Findings

#### Finding 1: Finding #4 — The Freshness Multiplier
* **Paper Statement:** Evaluates growth-to-decline ratios across freshness windows. The 31-90 day window shows a 7.88:1 growth-to-decline ratio, while the 361+ bucket shows a 283:1 spike.
* **Methodology Question:** *"What is the exact sample size $n$ in the 361+ freshness bucket, and how much of the observed 283:1 ratio is driven by survivor bias (i.e. only exceptionally stable pages surviving un-updated for over a year)?"*
* **Context & Rigor:** When evaluating content that has not been updated for over 361 days, pages that suffered traffic collapse may have been pruned or deleted by site managers. A high growth-to-decline ratio in an un-updated cohort can reflect survivor bias rather than evidence that leaving content untouched promotes growth.

#### Finding 2: Finding #9 — Captured Traffic Value
* **Paper Statement:** Recommends `clicks × CPC` ($253.5K) as a defensible value proxy over `impressions × CPC` ($73.0M).
* **Methodology Question:** *"How are missing CPC values handled for informational pages without paid search bidding data, and does applying average category CPC across non-transactional intents distort total valuation?"*
* **Context & Rigor:** While `clicks × CPC` is significantly more defensible than multiplying by raw impressions, search keywords for informational content often lack commercial CPC benchmarks. Imputing non-zero CPC values for high-volume informational queries risks overstating economic value relative to transactional pages.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show the table.*

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Load starter dataset
data_path = Path("../../data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# 2. Preprocessing & Feature Vector Construction
df["scroll_rate_filled"] = df["scroll_rate"].fillna(0)
df["engagement_rate_filled"] = df["engagement_rate"].fillna(0)
df["ai_traffic_pct_filled"] = df["ai_traffic_pct"].fillna(0)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["search_volume_filled"] = df["search_volume"].fillna(0)
df["competition_filled"] = df["competition"].fillna(0)
df["cpc_filled"] = df["cpc"].fillna(0)
df["word_count_filled"] = df["word_count"].fillna(df["word_count"].median())
df["stale_flag"] = (df["days_since_last_update"] >= 180).astype(int)
df["high_impression_flag"] = (df["impressions_90d"] >= 500).astype(int)
df["impressions_per_day"] = df["impressions_90d"] / (df["content_age_days"] + 1)

content_type_dummies = pd.get_dummies(df["content_type"], prefix="type", drop_first=True)
intent_dummies = pd.get_dummies(df["main_intent"].fillna("unknown"), prefix="intent", drop_first=True)

honest_numeric_cols = [
    "content_age_days", "days_since_last_update", "stale_flag",
    "impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d",
    "engaged_sessions_90d", "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate_filled", "scroll_rate_filled", "ai_traffic_pct_filled",
    "search_volume_filled", "competition_filled", "cpc_filled",
    "word_count_filled", "has_keyword_data", "has_word_count",
    "high_impression_flag", "impressions_per_day"
]

X = pd.concat([df[honest_numeric_cols], content_type_dummies, intent_dummies], axis=1)
X = X.loc[:, ~X.columns.duplicated()]
y = df["is_declining_label"]
groups = df["client_id"]

def precision_at_k(probs, y_true, k=50):
    top_k_idx = np.argsort(-probs)[:k]
    return y_true.iloc[top_k_idx].mean()

# 3. Validation Design 1: Standard Random Split (In-sample client mix)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.25, random_state=42)
clf_random = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_random.fit(X_train_r, y_train_r)
prob_random = clf_random.predict_proba(X_test_r)[:, 1]
acc_random = accuracy_score(y_test_r, clf_random.predict(X_test_r))
p50_random = precision_at_k(prob_random, y_test_r, k=50)

# 4. Validation Design 2: Honest Grouped Split (GroupKFold on client_id)
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

clf_grouped = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_grouped.fit(X_train_g, y_train_g)
prob_grouped = clf_grouped.predict_proba(X_test_g)[:, 1]
acc_grouped = accuracy_score(y_test_g, clf_grouped.predict(X_test_g))
p50_grouped = precision_at_k(prob_grouped, y_test_g, k=50)

# Before/After Comparison Table
split_comp_df = pd.DataFrame([
    {"Validation Split": "Random Train/Test Split", "Accuracy": f"{acc_random:.4f}", "Precision@50": f"{p50_random:.4f}", "Client Overlap": "Yes (Shared clients)"},
    {"Validation Split": "Honest Grouped Split (GroupKFold)", "Accuracy": f"{acc_grouped:.4f}", "Precision@50": f"{p50_grouped:.4f}", "Client Overlap": "No (Unseen clients)"}
])

print("=== VALIDATION SPLIT COMPARISON (BEFORE vs AFTER) ===")
print(split_comp_df.to_string(index=False))

=== VALIDATION SPLIT COMPARISON (BEFORE vs AFTER) ===
                 Validation Split Accuracy Precision@50       Client Overlap
          Random Train/Test Split   0.6625       0.7800 Yes (Shared clients)
Honest Grouped Split (GroupKFold)   0.5755       0.5400  No (Unseen clients)


### Validation Design Audit Findings

* **The Memorization Gap:** Random train/test splits allow rows from the same client to appear in both training and test sets. The model can memorize client-specific traffic signatures, resulting in an artificially inflated Precision@50 (`0.5800`).
* **Honest Client-Holdout Generalization:** Evaluating under `GroupKFold` on `client_id` tests how well the model predicts declining content on entirely unseen websites. Precision@50 drops to `0.5400`, reflecting honest real-world performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# 1. Feature Leakage Verification
forbidden_columns = ["trend_direction", "trend_pct", "content_id", "client_id", "provider_used", "model_used"]
leaked_present = [col for col in forbidden_columns if col in X.columns]

print(f"Forbidden columns present in feature matrix X: {leaked_present} (Must be empty!)")
assert len(leaked_present) == 0, "Target leakage detected in feature vector!"

# 2. Error Analysis: Inspect 2 Real Failure Cases
test_df = df.iloc[test_idx].copy()
test_df["pred_prob"] = prob_grouped
test_df["pred_label"] = (prob_grouped >= 0.5).astype(int)

false_positives = test_df[(test_df["pred_label"] == 1) & (test_df["is_declining_label"] == 0)]
false_negatives = test_df[(test_df["pred_label"] == 0) & (test_df["is_declining_label"] == 1)]

print(f"\nTotal Test Error Count: False Positives = {len(false_positives):,}, False Negatives = {len(false_negatives):,}")

print("\n--- Error Example 1: False Positive (Flagged declining, but remained stable) ---")
if len(false_positives) > 0:
    fp = false_positives.iloc[0]
    print(f"Content ID: {fp['content_id']} | Age: {fp['content_age_days']}d | Update Age: {fp['days_since_last_update']}d | Impressions: {fp['impressions_90d']} | Avg Pos: {fp['avg_position']} | Predicted Prob: {fp['pred_prob']:.4f}")

print("\n--- Error Example 2: False Negative (Predicted stable, but suffered trend decline) ---")
if len(false_negatives) > 0:
    fn = false_negatives.iloc[0]
    print(f"Content ID: {fn['content_id']} | Age: {fn['content_age_days']}d | Update Age: {fn['days_since_last_update']}d | Impressions: {fn['impressions_90d']} | Avg Pos: {fn['avg_position']} | Predicted Prob: {fn['pred_prob']:.4f}")

Forbidden columns present in feature matrix X: [] (Must be empty!)

Total Test Error Count: False Positives = 1,957, False Negatives = 1,018

--- Error Example 1: False Positive (Flagged declining, but remained stable) ---
Content ID: content_d8ee6cc6d642 | Age: 329d | Update Age: 104d | Impressions: 20919 | Avg Pos: 2.2 | Predicted Prob: 0.5383

--- Error Example 2: False Negative (Predicted stable, but suffered trend decline) ---
Content ID: content_761a44afda12 | Age: 421d | Update Age: 22d | Impressions: 9449 | Avg Pos: 7.3 | Predicted Prob: 0.3904


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim Audit & Public-Safe Rewrite

* **Uncautious / Overconfident Claim:**
  > *"Our artificial intelligence model predicts Google algorithm drops with near 100% accuracy and guarantees which pages will recover traffic once refreshed."*

* **Public-Safe Claim Rewrite:**
  > *"Under an honest client-holdout validation design (`GroupKFold`), our decision-support scoring model achieved a measured **Precision@50 of 0.5400** on observed 90-day search performance trends. The model provides a directional review queue for content teams to prioritize pages for manual audit, rather than causal predictions of search engine algorithms."*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.